# utils_test

This notebook (utils_test) is a utility test harness for `pytorch2ltspice.utils`.
It generates PyTorch models from switch-based definitions, exports standalone `.py` classes and
LTspice subcircuits `.sp`, and runs parity checks between PyTorch and LTspice outputs.
This notebook serves as both a regression test and a usage example for new users.

---

## Quick Start

1. Set `ENV_NAME` and `MODEL_NAME` in the Configuration section.
2. Run `step1()` to generate the model class and subcircuit.
3. Run `step2()` and `step3()` to simulate and compare outputs.

---

---


## Change Log:

2025-09-21, Initial Version

2025-10-01,
- Python Code Generator: added `clone_state` method in generated classes for safe hidden state duplication.
- Python Code Generator: updated `forward` in generated classes to accept h as an alias for state.

2025-12-14,
- Updated the notebook to use the lowercase pytorch2ltspice module name and removed relative-path imports.

2025-12-29,
- Renamed ModelGen.ipynb to utils_test.ipynb to reflect the move of utilities into pytorch2ltspice.utils in v0.1.2.

2025-01-03,
- Removed SUBCKT_NAME parameter.
---

In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from torch import nn

from PyLTSpice import LTspice, RawRead, SimRunner

from pytorch2ltspice import export_model_to_ltspice
from pytorch2ltspice.utils.modelgen import build_model_from_sequential
from pytorch2ltspice.utils.sampling import sample_on_clock
from pytorch2ltspice.utils.siggen import generate_siggen_asc_asy


---
## Configuration

In [ ]:
# Environment configuration
# Add entries to ENV_CONFIG with required keys: "in" and "out"
ENV_NAME = "env_buck_9x2"              # Select environment file here
ENV_CONFIG = {
    "env_buck_9x1":  {"in": 9, "out": 1},
    "env_buck_9x2":  {"in": 9, "out": 2},
    # Add more environments as needed
}

# Model configuration
# Add entries to MODEL_CONFIG with required key: "clk_needed"
MODEL_NAME = "mlp"                     # "mlp"/"rnn_linear"/"gru_linear"/"linear_lstm_linear"
MODEL_CONFIG = {
    "mlp":  {"clk_needed": False},
    "rnn_linear":  {"clk_needed": True},
    "gru_linear":  {"clk_needed": True},
    "linear_lstm_linear":  {"clk_needed": True},
    # Add more models as needed
}


# LTspice simulation configuration
SIM_STEP = 200                         # Number of time steps to run in LTSpice 
SIM_TIMEOUT = 300                      # Timeout therhold in seconds

# Working directory for pyLTspice
NOTEBOOK_DIR = Path.cwd()
ENVDIR  = NOTEBOOK_DIR / "gym"
OUTDIR  = NOTEBOOK_DIR / "gym"         # save at same directory as .asc file
WORKDIR = NOTEBOOK_DIR / "tmp" 
WORKDIR.mkdir(exist_ok=True)

---
## Model Definition

In [ ]:
def make_model(model_name: str) -> nn.Sequential:
    if model_name == "mlp":
        return nn.Sequential(
            nn.Linear(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
            nn.Tanh(),
        )
    elif model_name == "rnn_linear":
        return nn.Sequential(
            nn.RNNCell(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
            nn.Tanh(),
        )
    elif model_name == "gru_linear":
        return nn.Sequential(
            nn.GRUCell(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
            nn.Tanh(),
        )
    elif model_name == "linear_lstm_linear":
        return nn.Sequential(
            nn.Linear(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.Tanh(),
            nn.LSTMCell(32, 32),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
            nn.Tanh(),
        )
    # Example extension for stacked LSTM cells:
    # elif model_name == "stacked_lstm":
    #     return nn.Sequential(
    #         nn.LSTMCell(ENV_CONFIG[ENV_NAME]["in"], 32),
    #         nn.LSTMCell(32, 32),
    #         nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
    #         nn.Tanh(),
    #     )
    else:
        raise ValueError(f"Unknown model preset: {model_name}")


---
## Step0) Global variables for Step1-3

In [ ]:
DEVICE = torch.device('cpu')

---
## Step1) Create Python code and LTspice sub-circuit

In [ ]:
def step1() -> nn.modules:
    # Select input sequential model
    seq = make_model(MODEL_NAME)

    # Generate & save python code via utils
    GenClass = build_model_from_sequential(
        MODEL_NAME,
        seq,
        out_dir=ENVDIR,
        out_py_name=PY_FILENAME,
    )

    # Instantiate the generated class
    actor = GenClass().to(DEVICE)

    # Export LTSpice sub-circuit
    export_model_to_ltspice(actor.model, filename=f"{OUTDIR}/{SP_FILENAME}.sp", subckt_name=MODEL_NAME, verbose=False)

    return actor


## Step2) Run Simulation on LTspice and Python

Note: This notebook assumes the env `.asc` file wires `sig_gen*.sp` into the circuit to produce `NNOUT*n` nodes.
If not, include the generated `sig_gen*.sp` in your environment schematic or parameter file.


In [ ]:
def step2(module):
    # 0) Create random noise generator asc/asy
    NOISE_SIGMA = 0.01
    noises = np.zeros((SIM_STEP, ENV_CONFIG[ENV_NAME]["out"]), dtype=float)
    for i in range(ENV_CONFIG[ENV_NAME]["out"]):
        noises[:, i] = np.random.normal(loc=0.0, scale=NOISE_SIGMA, size=SIM_STEP)
        generate_siggen_asc_asy(
            signals=noises[:, i],
            asc_path=f"{ENVDIR}/sig_gen{i+1}.asc",
            gen_symbol=True,
            subckt_name=f"sig_gen{i+1}",
        )

    # 1) Create parameter file
    with open(f"{ENVDIR}/{ENV_NAME}_param.txt", 'w', encoding='utf-8') as f:
        f.write(f".param STEPS={SIM_STEP}\n")
        nn_inputs = ' '.join(f'NNin{i+1}' for i in range(ENV_CONFIG[ENV_NAME]["in"]))
        nn_outputs = ' '.join(f'NNout{i+1}' for i in range(ENV_CONFIG[ENV_NAME]["out"]))
        ports = " ".join(p for p in [nn_inputs, ("ctrlclk" if MODEL_CONFIG[MODEL_NAME]["clk_needed"] else ""), nn_outputs] if p)
        f.write(f"X99 {ports} {MODEL_NAME}\n")
        f.write(f".include {SP_FILENAME}.sp\n")

    # 2) Create PyLTspice SimRunner instance at WORKDIR
    shutil.copy2(f"{OUTDIR}/{SP_FILENAME}.sp", f"{WORKDIR}/")  
    shutil.copy2(f"{ENVDIR}/{ENV_NAME}_param.txt", f"{WORKDIR}/")  
    runner = SimRunner(output_folder=WORKDIR, simulator=LTspice)
    netlist = runner.create_netlist(f"{ENVDIR}/{ENV_NAME}.asc")
        
    # 3) Run LTSpice simulation
    raw, log = runner.run_now(netlist, timeout=SIM_TIMEOUT)
    raw_data = RawRead(raw)
    df = raw_data.to_dataframe()
    df = sample_on_clock(df, clk='V(ctrlclk)')

    # 4) Extract states, actions
    states  = df[[f'V(nnin{i+1})' for i in range(ENV_CONFIG[ENV_NAME]["in"])]].values[:-1]          # S[t]
    nnouts  = df[[f'V(nnout{i+1})' for i in range(ENV_CONFIG[ENV_NAME]["out"])]].values[:-1]        # NNOUTx[t]
    actions  = df[[f'V(nnout{i+1}n)' for i in range(ENV_CONFIG[ENV_NAME]["out"])]].values[:-1]      # NNOUTx[t] + Noise[t]

    # 5) Clean PyLTspice files
    runner.cleanup_files()
    os.remove(f"{ENVDIR}/{ENV_NAME}.net")
    os.remove(f"{WORKDIR}/{SP_FILENAME}.sp")
    os.remove(f"{WORKDIR}/{ENV_NAME}_param.txt")

    # 5) Calculate PyTorch output using observation from LTspice
    states_t  = torch.tensor(states,  dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        nnouts_py = module(states_t).cpu().numpy()
        if nnouts_py.ndim == 1:
            nnouts_py = nnouts_py.reshape(-1, ENV_CONFIG[ENV_NAME]["out"])
    
    noises = noises[:nnouts_py.shape[0]]
    actions_py = nnouts_py + noises


    return nnouts, nnouts_py, actions, actions_py


## Step3) Compare outputs from PyTorch and LTspice

Note: `NNOUT*` is the raw model output, while `NNOUT*n` is the output with injected noise.


In [ ]:
def step3(nnouts, nnouts_py, actions, actions_py):
    #Plot Scatter graph
    fig = go.Figure()
    ltspice_o = np.asarray(nnouts)
    pytorch_o = np.asarray(nnouts_py)
    ltspice_on = np.asarray(actions)
    pytorch_on = np.asarray(actions_py)
    samples = ltspice_on.shape[0]
    x_axis = np.arange(samples)
    for idx in range(ltspice_on.shape[1]):
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=ltspice_o[:, idx],
            mode='markers',
            name=f'NNOUT{idx + 1}(LTspice)'
        ))
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=pytorch_o[:, idx],
            mode='markers',
            name=f'NNOUT{idx + 1}(PyTorch)'
        ))
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=ltspice_on[:, idx],
            mode='markers',
            name=f'NNOUT{idx + 1}n(LTspice)'
        ))
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=pytorch_on[:, idx],
            mode='markers',
            name=f'NNOUT{idx + 1}n(PyTorch)'
        ))
    fig.update_layout(
        title=f"ENV={ENV_NAME}<br>MODEL={MODEL_NAME}",
        xaxis_title='Sample index',
        yaxis_title='Output'
    )
    fig.show()

    #Print MAE/MSE 
    diff = nnouts - nnouts_py
    diffn = actions - actions_py
    mae_per_output = np.mean(np.abs(diff), axis=0)
    mse_per_output = np.mean(diff ** 2, axis=0)
    mae_per_outputn = np.mean(np.abs(diffn), axis=0)
    mse_per_outputn = np.mean(diffn ** 2, axis=0)
    for idx, (mae_val, mse_val, mae_valn, mse_valn) in enumerate(zip(mae_per_output, mse_per_output, mae_per_outputn, mse_per_outputn), start=1):
        print(f"  NNOUT{idx}: MAE={mae_val:.6f}, MSE={mse_val:.6f}")
        print(f"  NNOUT{idx}n: MAE={mae_valn:.6f}, MSE={mse_valn:.6f}")

---
## Execution

In [ ]:
ENV_NAME = "env_buck_9x2"
MODEL_NAME = "mlp"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)

In [ ]:
ENV_NAME = "env_buck_9x2"
MODEL_NAME = "rnn_linear"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)

In [ ]:
ENV_NAME = "env_buck_9x2"
MODEL_NAME = "gru_linear"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)

In [ ]:
ENV_NAME = "env_buck_9x2"
MODEL_NAME = "linear_lstm_linear"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)

In [ ]:
ENV_NAME = "env_buck_9x1"
MODEL_NAME = "mlp"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)

In [ ]:
ENV_NAME = "env_buck_9x1"
MODEL_NAME = "rnn_linear"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)

In [ ]:
ENV_NAME = "env_buck_9x1"
MODEL_NAME = "gru_linear"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)

In [ ]:
ENV_NAME = "env_buck_9x1"
MODEL_NAME = "linear_lstm_linear"
PY_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output python file name
SP_FILENAME = ENV_NAME + '_' + MODEL_NAME     # output ltspice subcircuit name
actor = step1()
nnouts, nnouts_py, actions, actions_py = step2(actor)
step3(nnouts, nnouts_py, actions, actions_py)